# Epoch analysis

CS+/Trace/US-specific activity for both channels: heatmaps, average-vs-individual-trial traces, and field width / trial reliability - toward K99 Aim 1 (astrocyte-neuron temporal coordination).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from CalciumKit.mouse import Mouse
from CalciumKit.trial_analysis import extract_trials, field_width, trial_reliability

sns.set_style("ticks")
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


In [ ]:
raw_root = Path(r"D:\ImagingData\Raw")
processed_root = Path(r"D:\ImagingData\Processed\Chandler")

mouse = Mouse("Chandler", raw_root, processed_root)
session = mouse.sessions["day2"]
session.task = "tfc"
ts = session.trial_structure

CHANNELS = [("red", "spks", "Reds", "#C62828", "Neurons"), ("green", "dff", "Greens", "#2E7D32", "Astrocytes")]
EPOCH_COLORS = {"CS+": "#2E7D32", "Trace": "0.3", "US": "#C62828"}

def epoch_windows():
    return {
        "CS+": (ts.tone_times, ts.tone_dur),
        "Trace": (ts.tone_times + ts.tone_dur, ts.trace_dur),
        "US": (ts.shock_times, ts.us_window),
    }

EPOCHS = list(epoch_windows().keys())

def epoch_trial_data(color, signal, epoch_name):
    tsd = session.tsd(color, signal)
    event_times, duration = epoch_windows()[epoch_name]
    return extract_trials(tsd.values, tsd.t, event_times, trial_window=duration, window_before=0, zscore=True)


## Epoch-specific heatmaps

Trial-averaged activity within each epoch alone (not the whole trial), cells sorted by their peak time within that epoch.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 5), dpi=200)

for row, (color, signal, cmap, main_color, label) in enumerate(CHANNELS):
    for col, epoch_name in enumerate(EPOCHS):
        trial_data, t_rel = epoch_trial_data(color, signal, epoch_name)
        avg = np.nanmean(trial_data, axis=1)
        order = np.argsort(np.argmax(avg, axis=1))

        ax = axes[row, col]
        ax.imshow(avg[order], aspect="auto", cmap=cmap, extent=[t_rel[0], t_rel[-1], avg.shape[0], 0])
        ax.set_title(f"{label} - {epoch_name}", fontsize=9, fontweight="bold", color=EPOCH_COLORS[epoch_name])
        ax.set_xlabel("Time in epoch (s)")
        ax.set_yticks([0, avg.shape[0]])
        ax.set_yticklabels(["1", str(avg.shape[0])])
        if col == 0:
            ax.set_ylabel("Cell (sorted)")

plt.tight_layout()


## Average trace vs individual trials

Population mean (across cells) for each individual trial, thin gray, with the overall trial-average in color.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 5), dpi=200, sharey="row")

for row, (color, signal, cmap, main_color, label) in enumerate(CHANNELS):
    for col, epoch_name in enumerate(EPOCHS):
        trial_data, t_rel = epoch_trial_data(color, signal, epoch_name)
        pop_trials = np.nanmean(trial_data, axis=0)  # trials x time
        n_trials = pop_trials.shape[0]
        trial_colors = plt.cm.viridis(np.linspace(0, 0.85, n_trials))

        ax = axes[row, col]
        for trial_i, trial in enumerate(pop_trials):
            ax.plot(t_rel, trial, color=trial_colors[trial_i], lw=1.5, label=f"Trial {trial_i + 1}")
        ax.set_title(f"{label} - {epoch_name}", fontsize=9, fontweight="bold", color=EPOCH_COLORS[epoch_name])
        ax.set_xlabel("Time in epoch (s)")
        if col == 0:
            ax.set_ylabel("Population activity\n(z-score)")
        if col == 2:
            ax.legend(frameon=False, fontsize=6, loc="upper right")
        sns.despine(ax=ax)

plt.tight_layout()


## Field width and trial reliability

Over CS+/Trace/US combined (tone onset through 10s past shock) - is astrocyte tuning broader than neuronal, and is either reliable across trials?

In [ ]:
def field_widths_and_reliability(color, signal):
    tsd = session.tsd(color, signal)
    window_end = ts.shock_offset + ts.us_window
    trial_data, t_rel = extract_trials(tsd.values, tsd.t, ts.tone_times, trial_window=window_end, window_before=0, zscore=True)

    avg = np.nanmean(trial_data, axis=1)
    widths = np.array([field_width(pd.Series(avg[c], index=t_rel)) for c in range(avg.shape[0])])
    reliability = trial_reliability(trial_data)
    return widths, reliability


results = {label: field_widths_and_reliability(color, signal) for color, signal, cmap, main_color, label in CHANNELS}

df = pd.concat([
    pd.DataFrame({"width": results[label][0], "reliability": results[label][1], "cell_type": label})
    for _, _, _, _, label in CHANNELS
])
PALETTE = {"Neurons": "#C62828", "Astrocytes": "#2E7D32"}

fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6), dpi=200)

sns.boxplot(data=df, x="cell_type", y="width", ax=axes[0], color="white", fliersize=0, width=0.5, linewidth=0.8)
sns.stripplot(data=df, x="cell_type", y="width", ax=axes[0], hue="cell_type", palette=PALETTE, alpha=0.5, size=3, jitter=0.25, legend=False)
axes[0].set_yscale("log")
axes[0].set_ylabel("Field width (s)")
axes[0].set_xlabel("")

sns.boxplot(data=df, x="cell_type", y="reliability", ax=axes[1], color="white", fliersize=0, width=0.5, linewidth=0.8)
sns.stripplot(data=df, x="cell_type", y="reliability", ax=axes[1], hue="cell_type", palette=PALETTE, alpha=0.5, size=3, jitter=0.25, legend=False)
axes[1].set_ylabel("Trial reliability (r)")
axes[1].set_xlabel("")

for label, color in PALETTE.items():
    sub = df[df["cell_type"] == label]
    axes[2].scatter(sub["width"], sub["reliability"], color=color, alpha=0.5, s=10, label=label)
axes[2].set_xscale("log")
axes[2].set_xlabel("Field width (s)")
axes[2].set_ylabel("Trial reliability (r)")
axes[2].legend(frameon=False, fontsize=6)

for ax in axes:
    sns.despine(ax=ax)

plt.tight_layout()


## Control: dF/F for both channels

The comparison above uses `spks` (deconvolved) for neurons and `dff` for astrocytes - different signal types, not just different cell types. This repeats it with `dff` for both, to see how much of the width/reliability gap survives once the signal type is matched.

In [ ]:
CHANNELS_DFF = [("red", "dff", "Reds", "#C62828", "Neurons (dF/F)"), ("green", "dff", "Greens", "#2E7D32", "Astrocytes")]

results_dff = {label: field_widths_and_reliability(color, signal) for color, signal, cmap, main_color, label in CHANNELS_DFF}

df_dff = pd.concat([
    pd.DataFrame({"width": results_dff[label][0], "reliability": results_dff[label][1], "cell_type": label})
    for _, _, _, _, label in CHANNELS_DFF
])
PALETTE_DFF = {"Neurons (dF/F)": "#C62828", "Astrocytes": "#2E7D32"}

fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6), dpi=200)

sns.boxplot(data=df_dff, x="cell_type", y="width", ax=axes[0], color="white", fliersize=0, width=0.5, linewidth=0.8)
sns.stripplot(data=df_dff, x="cell_type", y="width", ax=axes[0], hue="cell_type", palette=PALETTE_DFF, alpha=0.5, size=3, jitter=0.25, legend=False)
axes[0].set_yscale("log")
axes[0].set_ylabel("Field width (s)")
axes[0].set_xlabel("")

sns.boxplot(data=df_dff, x="cell_type", y="reliability", ax=axes[1], color="white", fliersize=0, width=0.5, linewidth=0.8)
sns.stripplot(data=df_dff, x="cell_type", y="reliability", ax=axes[1], hue="cell_type", palette=PALETTE_DFF, alpha=0.5, size=3, jitter=0.25, legend=False)
axes[1].set_ylabel("Trial reliability (r)")
axes[1].set_xlabel("")

for label, color in PALETTE_DFF.items():
    sub = df_dff[df_dff["cell_type"] == label]
    axes[2].scatter(sub["width"], sub["reliability"], color=color, alpha=0.5, s=10, label=label)
axes[2].set_xscale("log")
axes[2].set_xlabel("Field width (s)")
axes[2].set_ylabel("Trial reliability (r)")
axes[2].legend(frameon=False, fontsize=6)

for ax in axes:
    sns.despine(ax=ax)

plt.tight_layout()


## Control: dF/F for both channels

The comparison above uses `spks` (deconvolved) for neurons and `dff` for astrocytes - different signal types, not just different cell types. This repeats it with `dff` for both, to see how much of the width/reliability gap survives once the signal type is matched.

In [ ]:
CHANNELS_DFF = [("red", "dff", "Reds", "#C62828", "Neurons (dF/F)"), ("green", "dff", "Greens", "#2E7D32", "Astrocytes")]

results_dff = {label: field_widths_and_reliability(color, signal) for color, signal, cmap, main_color, label in CHANNELS_DFF}

df_dff = pd.concat([
    pd.DataFrame({"width": results_dff[label][0], "reliability": results_dff[label][1], "cell_type": label})
    for _, _, _, _, label in CHANNELS_DFF
])
PALETTE_DFF = {"Neurons (dF/F)": "#C62828", "Astrocytes": "#2E7D32"}

fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6), dpi=200)

sns.boxplot(data=df_dff, x="cell_type", y="width", ax=axes[0], color="white", fliersize=0, width=0.5, linewidth=0.8)
sns.stripplot(data=df_dff, x="cell_type", y="width", ax=axes[0], hue="cell_type", palette=PALETTE_DFF, alpha=0.5, size=3, jitter=0.25, legend=False)
axes[0].set_yscale("log")
axes[0].set_ylabel("Field width (s)")
axes[0].set_xlabel("")

sns.boxplot(data=df_dff, x="cell_type", y="reliability", ax=axes[1], color="white", fliersize=0, width=0.5, linewidth=0.8)
sns.stripplot(data=df_dff, x="cell_type", y="reliability", ax=axes[1], hue="cell_type", palette=PALETTE_DFF, alpha=0.5, size=3, jitter=0.25, legend=False)
axes[1].set_ylabel("Trial reliability (r)")
axes[1].set_xlabel("")

for label, color in PALETTE_DFF.items():
    sub = df_dff[df_dff["cell_type"] == label]
    axes[2].scatter(sub["width"], sub["reliability"], color=color, alpha=0.5, s=10, label=label)
axes[2].set_xscale("log")
axes[2].set_xlabel("Field width (s)")
axes[2].set_ylabel("Trial reliability (r)")
axes[2].legend(frameon=False, fontsize=6)

for ax in axes:
    sns.despine(ax=ax)

plt.tight_layout()
